# Revenue and Income Statistics by Museum Type

For each museum type, compute count, mean, median, min, max, and standard deviation of Revenue and Income.

Self-contained — needs only `museums.csv` in the same folder. Run **Kernel → Restart Kernel and Run All Cells**.

## Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("museums.csv", low_memory=False)
df = df.rename(columns={"Museum Type": "museum_type"})
df["museum_type"] = df["museum_type"].str.title()
print(f"Loaded {len(df):,} museums")

Loaded 33,072 museums


## Step 1: Clean Revenue and Income

About 32% of rows show exactly `0` for these columns. Form 990 filers reporting genuine $0 of revenue or income would be unusual; these zeros are sentinel values for "not reported" (e.g., from 990-N postcards that omit dollar amounts). Convert them to `NaN` so they don't drag down the statistics.

In [2]:
for col in ["Revenue", "Income"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    n_zero = (df[col] == 0).sum()
    df[col] = df[col].where(df[col] != 0, np.nan)
    print(f"{col}: converted {n_zero:,} sentinel zeros to NaN; "
          f"{df[col].notna().sum():,} usable values remain")

Revenue: converted 10,783 sentinel zeros to NaN; 11,507 usable values remain
Income: converted 10,740 sentinel zeros to NaN; 12,221 usable values remain


## Step 2: Methodology note — parent organizations

Many museums share a Form 990 (and an EIN) with their parent organization — typically a university. Harvard files a single 990 for its 20 museums; that filing reports Harvard's entire $5.8B institutional revenue, which then appears as the Revenue value for every Harvard museum row across multiple museum types.

This means:
- **Maxes** at the high end are typically parent-university financials, not museum financials.
- **Means** are dragged upward by these inflated values.
- **Standard deviations** are huge for the same reason.
- **Medians** are robust to the inflation — they're the most reliable measure of "typical" revenue for a museum of a given type.

The tables below report all six measures as requested, but the discussion at the end focuses on medians.

## Step 3: Revenue statistics by museum type

In [3]:
rev_stats = (df.groupby("museum_type")["Revenue"]
               .agg(["count", "mean", "median", "min", "max", "std"])
               .sort_values("median", ascending=False))
rev_stats

,count,mean,median,min,max,std
museum_type,,,,,,
Art Museum,1558,1.192889e+08,5689161.0,-106157.0,5.840349e+09,4.574068e+08
Science & Technology Museum Or Planetarium,312,1.509469e+08,1076971.5,-1723674.0,5.840349e+09,7.570478e+08
"Zoo, Aquarium, Or Wildlife Conservation",215,7.447497e+06,992041.0,1668.0,1.210595e+08,1.593914e+07
Natural History Museum,162,1.240253e+08,637259.0,5.0,5.840349e+09,6.709189e+08
"Arboretum, Botanical Garden, Or Nature Center",499,1.158044e+08,584793.0,-1428.0,5.840349e+09,6.534290e+08
Children'S Museum,263,1.674879e+06,437097.0,4150.0,3.702075e+07,4.140334e+06
General Museum,2321,4.756831e+07,178521.0,-2127393.0,5.840349e+09,4.134059e+08
History Museum,1250,1.526204e+07,126117.0,-567630.0,5.121523e+09,2.115809e+08
Historic Preservation,4927,5.115653e+06,81008.0,-83151.0,5.840349e+09,1.429799e+08


### Same table, formatted as currency

In [4]:
money_cols = ["mean", "median", "min", "max", "std"]
(rev_stats.style
    .format({c: "${:,.0f}" for c in money_cols})
    .format({"count": "{:,.0f}"})
    .set_caption("Revenue by museum type (sorted by median)"))

,count,mean,median,min,max,std
museum_type,,,,,,
Art Museum,"1,558",119288853.135430,5689161.000000,-106157.000000,5840349457.000000,457406770.959569
Science & Technology Museum Or Planetarium,312,150946908.413462,1076971.500000,-1723674.000000,5840349457.000000,757047816.487348
"Zoo, Aquarium, Or Wildlife Conservation",215,7447496.730233,992041.000000,1668.000000,121059457.000000,15939142.488651
Natural History Museum,162,124025346.333333,637259.000000,5.000000,5840349457.000000,670918866.136343
"Arboretum, Botanical Garden, Or Nature Center",499,115804374.679359,584793.000000,-1428.000000,5840349457.000000,653429000.254147
Children'S Museum,263,1674879.007605,437097.000000,4150.000000,37020754.000000,4140334.135047
General Museum,"2,321",47568310.480396,178521.000000,-2127393.000000,5840349457.000000,413405860.647639
History Museum,"1,250",15262043.864000,126117.000000,-567630.000000,5121523000.000000,211580867.122735
Historic Preservation,"4,927",5115653.092551,81008.000000,-83151.000000,5840349457.000000,142979931.554360


## Step 4: Income statistics by museum type

In [5]:
inc_stats = (df.groupby("museum_type")["Income"]
               .agg(["count", "mean", "median", "min", "max", "std"])
               .sort_values("median", ascending=False))
inc_stats

,count,mean,median,min,max,std
museum_type,,,,,,
Art Museum,1635,3.757380e+08,7510681.0,1.0,8.318144e+10,3.758309e+09
Science & Technology Museum Or Planetarium,321,1.248100e+09,1115335.0,100.0,8.318144e+10,9.292070e+09
"Zoo, Aquarium, Or Wildlife Conservation",225,8.661270e+06,1004533.0,1.0,1.374917e+08,1.940326e+07
"Arboretum, Botanical Garden, Or Nature Center",535,6.972698e+08,826511.0,9.0,8.318144e+10,6.446308e+09
Natural History Museum,172,8.367585e+08,662924.5,5.0,8.318144e+10,6.710489e+09
Children'S Museum,268,2.418091e+06,448785.5,1.0,1.040673e+08,9.699586e+06
General Museum,2628,2.918066e+08,207100.0,1.0,8.318144e+10,4.596182e+09
History Museum,1334,2.688157e+07,147327.5,-923.0,1.065060e+10,4.267242e+08
Historic Preservation,5103,2.328798e+07,94887.0,1.0,8.318144e+10,1.184123e+09


In [6]:
(inc_stats.style
    .format({c: "${:,.0f}" for c in money_cols})
    .format({"count": "{:,.0f}"})
    .set_caption("Income by museum type (sorted by median)"))

,count,mean,median,min,max,std
museum_type,,,,,,
Art Museum,"1,635",375738027.727829,7510681.000000,1.000000,83181439574.000000,3758308531.794097
Science & Technology Museum Or Planetarium,321,1248099524.834891,1115335.000000,100.000000,83181439574.000000,9292070345.875931
"Zoo, Aquarium, Or Wildlife Conservation",225,8661269.568889,1004533.000000,1.000000,137491679.000000,19403258.004452
"Arboretum, Botanical Garden, Or Nature Center",535,697269810.016822,826511.000000,9.000000,83181439574.000000,6446308472.277184
Natural History Museum,172,836758488.151163,662924.500000,5.000000,83181439574.000000,6710489206.787044
Children'S Museum,268,2418091.022388,448785.500000,1.000000,104067302.000000,9699585.870901
General Museum,"2,628",291806591.460046,207100.000000,1.000000,83181439574.000000,4596182098.054888
History Museum,"1,334",26881571.676162,147327.500000,-923.000000,10650597555.000000,426724192.159435
Historic Preservation,"5,103",23287983.684499,94887.000000,1.000000,83181439574.000000,1184123344.537499


## Step 5: Medians side-by-side

The most readable summary — robust to parent-university distortion.

In [7]:
medians = pd.DataFrame({
    "museums (n)": rev_stats["count"].astype(int),
    "median revenue": rev_stats["median"],
    "median income":  inc_stats["median"],
}).sort_values("median revenue", ascending=False)

(medians.style
    .format({"museums (n)": "{:,}",
             "median revenue": "${:,.0f}",
             "median income":  "${:,.0f}"})
    .set_caption("Median revenue and income by museum type"))

,museums (n),median revenue,median income
museum_type,,,
Art Museum,"1,558","$5,689,161","$7,510,681"
Science & Technology Museum Or Planetarium,312,"$1,076,972","$1,115,335"
"Zoo, Aquarium, Or Wildlife Conservation",215,"$992,041","$1,004,533"
Natural History Museum,162,"$637,259","$662,924"
"Arboretum, Botanical Garden, Or Nature Center",499,"$584,793","$826,511"
Children'S Museum,263,"$437,097","$448,786"
General Museum,"2,321","$178,521","$207,100"
History Museum,"1,250","$126,117","$147,328"
Historic Preservation,"4,927","$81,008","$94,887"


## Patterns

**Reading the medians (the trustworthy column):**

- **Art Museums lead by a wide margin** with a median revenue of **~$5.7M** — roughly 5× the next category. Art museums are typically larger institutions with significant endowments, permanent collections requiring climate-controlled facilities, and concentrated cultural philanthropy.
- A **mid-tier** of operationally intensive institutions follows: Science & Technology (~$1.1M), Zoos/Aquariums (~$1.0M), and Natural History (~$640K). These have meaningful physical-plant requirements (planetariums, animal care, specimen storage).
- **Arboreta/Botanical Gardens** (~$585K) and **Children's Museums** (~$437K) are the next tier — both are usually purpose-built attractions with recurring operating costs.
- **General Museums** (~$179K) and **History Museums** (~$126K) sit lower — many are county or regional institutions with modest budgets.
- **Historic Preservation** is at the bottom with a median of just **~$81K**, despite being by far the largest category (4,927 filers). This category is dominated by tiny historical societies and small preserved sites that operate on volunteer labor and modest annual budgets.

**The order of medians tells a story:** revenue tracks roughly with operational complexity and visitor-attraction status — art and science draw philanthropy and admissions; historic houses and small society museums draw neither at scale.

**Why means and standard deviations are distorted:**

Compare Art Museums (median $5.7M, mean $119M) or Natural History (median $637K, mean $124M). The mean is 20–200× the median in most categories. This is the parent-organization effect: Harvard files a single 990 reporting $5.8B in revenue, and that figure attaches to every Harvard museum row across multiple categories. The same is true for Stanford, Yale, NYU, Johns Hopkins, and dozens of other universities. Means and maxes are dragged upward by these inflated values, and standard deviations balloon to hundreds of millions.

**Income vs revenue:**

On Form 990, "Total Revenue" captures inflows from operations and contributions; "Total Income" is broader, including investment returns. For most small museums Income ≈ Revenue. The big gaps (Harvard reports $5.8B revenue but $83B income) come from large endowment investment portfolios — another signal that the high end of these distributions is parent organization data, not museum data.

**Practical takeaway:**

If you need a defensible "typical museum revenue" by type, use the **median** column. Means, maxes, and standard deviations in this dataset are unreliable for museum-level analysis because of the parent-organization issue.